In [3]:

import requests
import mlflow
from pipelines.data_preprocessing import (
    chunk_text,
    load_text,
    save_chunks_to_jsonl,
)
from pipelines.embeddings import compute_embeddings
from pipelines.rag_pipeline import (
    answer_query,
    build_rag_index,
    llm_answer_query,
    log_query,
    log_rag_version,
)


%load_ext autoreload
%autoreload 2


mlflow_uri = None

for uri in ["http://localhost:5000", "http://mlflow:5000"]:
    try:
        r = requests.get(f"{uri}/api/2.0/mlflow/experiments/search?max_results=1", timeout=2)
        if r.status_code == 200:
            mlflow_uri = uri 
            break
    except requests.RequestException:
        continue

if mlflow_uri is None:
    raise RuntimeError("MLflow сервер недоступен по обоим адресам")

mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("RAG_Experiment")

text_url = "https://blog.dzencode.com/ru/illyuziya-kachestva-vash-sayt-idealen-pozdravlyaem-vy-tolko-chto-sozhgli-byudzhet/"
article_text = load_text(text_url)

chunk_size =300
model_name="all-MiniLM-L6-v2"
chunks = chunk_text(article_text, chunk_size, semantic=False)

with mlflow.start_run(run_name="SentenceTransformers_v1"):
    
    chunks_with_embeddings=compute_embeddings(chunks, model_name)
    save_chunks_to_jsonl(chunks_with_embeddings)

    host = "localhost"
    port = 6333
    try:
        r = requests.get(f"http://{host}:{port}", timeout=2)
        r.raise_for_status()
    except Exception:
        host = "qdrant"
    
    index = build_rag_index(chunks_with_embeddings,host=host, port=port)
    mlflow.log_param("embedding_model", model_name)
    mlflow.log_param("chunk_size", chunk_size)

    query = "Что значит «Иллюзия качества»?"
    answer = answer_query(index, query)
    print(answer)


    %load_ext autoreload
    %autoreload 2

    query = "Что автор имеет в виду под 'иллюзией качества'?"
    llm_answer = llm_answer_query(query, answer)
    rag_file_path = "artifacts/rag_article.jsonl"
    print("Ответ LLM:", llm_answer)
    log_query(query ,llm_answer,"logs/query_log.json")
    log_rag_version(rag_file_path, "version_log.json")


    mlflow.log_artifact(rag_file_path)          # RAG-файл

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2025/09/17 09:44:59 INFO mlflow.tracking.fluent: Experiment with name 'RAG_Experiment' does not exist. Creating a new experiment.
Batches: 100%|██████████| 3/3 [00:07<00:00,  2.51s/it]


Сгенерировано 73 эмбеддингов, размерность 384
✅ RAG-файл сохранён: artifacts/rag_article.jsonl
В Qdrant загружено 73 чанков в коллекцию 'dzencode_articles'
Ваш проект болен “Иллюзией качества?”
Ответьте честно на три вопроса “Да” или “Нет”:
На совещаниях по проекту слово “красиво” звучит чаще, чем слово “конверсия”?
На совещаниях по проекту слово “красиво” звучит чаще, чем слово “конверсия”? P.S. Клуб анонимных перфекционистов
Кстати, о “памятниках”. Какой самый вопиющий пример “Иллюзии качества” вы встречали в своей жизни? Расскажите в комментариях — можно анонимно. и у клиента приятную иллюзию контроля и полного участия в процессе.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Ответ LLM: Автор, вероятно, имеет в виду «иллюзию качества» в смысле концепции психологии и управления.project management, которая описывает ситуацию, когда люди уделяют приоритетное внимание процессу создания something (например, разработки продукта, написанию кода, со